In [1]:
import os
import joblib
import geopandas as gpd
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, RandomizedSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    classification_report,
    make_scorer
)

In [2]:
# ==============================================================================
# 1. LOAD PRODUCTION FEATURE DATASET
# ==============================================================================
full_feat_path = os.path.join("../Datasets", "RZLT_Samples", "RZLT_AllCodes_10to1_FullFeatures.geojson")
if not os.path.exists(full_feat_path):
    full_feat_path = "RZLT_AllCodes_10to1_FullFeatures.geojson"

gdf = gpd.read_file(full_feat_path)

num_cols = [
    'SITE_AREA', 'LOG_SITE_AREA',
    'SHAPE_COMPACTNESS', 'PERIM_TO_AREA',
    'TRUE_PARCELS_WITHIN_500M', 'TRUE_PARCELS_WITHIN_1000M', 'LOG_TRUE_PARCELS_500M',
    'DIST_TO_BUS_M', 'DIST_TO_RAIL_LUAS_M', 'DIST_TO_ANY_TRANSIT_M',
    'BUS_STOPS_WITHIN_500M', 'LOG_DIST_BUS', 'LOG_DIST_RAIL_LUAS'
]
num_cols = [c for c in num_cols if c in gdf.columns]
cat_cols = ['ZONE_GZT']

gdf['ZONE_GZT'] = gdf['ZONE_GZT'].fillna("Unknown").astype(str).str.strip()
gdf['LOCAL_AUTHORITY'] = gdf['LOCAL_AUTHORITY'].fillna("Unknown").astype(str).str.strip()

X_df = gdf[num_cols + cat_cols].copy()
y = gdf['target'].values
groups = gdf['LOCAL_AUTHORITY'].values

In [3]:
# ==============================================================================
# 2. HYPERPARAMETER TUNING (OPTIMIZING MINORITY EXCLUSION PR-AUC)
# ==============================================================================
# Custom scorer targeting Minority Exclusion Class (0) PR-AUC
def exclusion_pr_auc(y_true, y_prob):
    return average_precision_score(1 - y_true, 1 - y_prob)

excl_scorer = make_scorer(exclusion_pr_auc, needs_proba=True)

# Encode full feature matrix for RandomizedSearchCV
ohe_search = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_encoded = np.hstack([X_df[num_cols].values, ohe_search.fit_transform(X_df[cat_cols])])

global_spw = (y == 0).sum() / (y == 1).sum()

param_dist = {
    'n_estimators': [200, 300, 400, 500],
    'learning_rate': [0.02, 0.03, 0.05, 0.08],
    'max_depth': [3, 4, 5, 6],
    'min_child_weight': [1, 3, 5, 8],
    'subsample': [0.75, 0.85, 0.95],
    'colsample_bytree': [0.70, 0.85, 1.0],
    'reg_alpha': [0.0, 0.1, 0.5, 1.0],
    'reg_lambda': [1.0, 2.0, 5.0]
}

base_xgb = XGBClassifier(
    scale_pos_weight=global_spw,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1
)

skf_tune = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = RandomizedSearchCV(
    estimator=base_xgb,
    param_distributions=param_dist,
    n_iter=30,
    scoring=excl_scorer,
    cv=skf_tune,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
search.fit(X_encoded, y)
best_params = search.best_params_

print("\n=== Best Hyperparameters Found ===")
for k, v in best_params.items():
    print(f"  {k}: {v}")

Fitting 5 folds for each of 30 candidates, totalling 150 fits


C:\Users\musab\AppData\Roaming\Python\Python313\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(



=== Best Hyperparameters Found ===
  subsample: 0.95
  reg_lambda: 1.0
  reg_alpha: 0.0
  n_estimators: 300
  min_child_weight: 1
  max_depth: 4
  learning_rate: 0.05
  colsample_bytree: 1.0


In [4]:
# ==============================================================================
# 3. EVALUATE TUNED MODEL ON STRATIFIED CV & STRATIFIED GROUP K-FOLD
# ==============================================================================
def run_validated_cv(cv_splitter, label, use_groups=False):
    oof_probs = np.zeros(len(y))
    fold_metrics = []
    splits = cv_splitter.split(X_df, y, groups) if use_groups else cv_splitter.split(X_df, y)

    for fold, (train_idx, val_idx) in enumerate(splits, start=1):
        X_tr_raw, X_va_raw = X_df.iloc[train_idx], X_df.iloc[val_idx]
        y_tr, y_va = y[train_idx], y[val_idx]

        ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        X_tr = np.hstack([X_tr_raw[num_cols].values, ohe.fit_transform(X_tr_raw[cat_cols])])
        X_va = np.hstack([X_va_raw[num_cols].values, ohe.transform(X_va_raw[cat_cols])])

        fold_spw = (y_tr == 0).sum() / (y_tr == 1).sum()

        clf = XGBClassifier(
            **best_params,
            scale_pos_weight=fold_spw,
            eval_metric='aucpr',
            random_state=42 + fold,
            n_jobs=-1
        )
        clf.fit(X_tr, y_tr)
        probs = clf.predict_proba(X_va)[:, 1]
        oof_probs[val_idx] = probs

        fold_metrics.append({
            'Fold': fold,
            'Val_Exclusions': int((y_va == 0).sum()),
            'ROC_AUC': roc_auc_score(y_va, probs),
            'PR_AUC_Inclusion(1)': average_precision_score(y_va, probs),
            'PR_AUC_Exclusion(0)': average_precision_score(1 - y_va, 1 - probs)
        })

    df_m = pd.DataFrame(fold_metrics)
    print(f"\n=== {label} ===")
    print(df_m.to_string(index=False))
    print("Mean Scores:")
    print(df_m[['ROC_AUC', 'PR_AUC_Inclusion(1)', 'PR_AUC_Exclusion(0)']].mean().round(4).to_string())

    prec_0, rec_0, thresholds = precision_recall_curve(1 - y, 1 - oof_probs)
    f1_0 = (2 * prec_0 * rec_0) / np.maximum(prec_0 + rec_0, 1e-8)
    best_idx = np.argmax(f1_0[:-1])
    opt_cutoff = 1.0 - thresholds[best_idx]
    print(f"Optimal Inclusion Cutoff: {opt_cutoff:.4f} | Max Exclusion F1: {f1_0[best_idx]:.4f}")
    return opt_cutoff

# 1) Standard Stratified 5-Fold CV
opt_threshold = run_validated_cv(
    StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    "Tuned Policy-Only: Stratified 5-Fold CV",
    use_groups=False
)

# 2) StratifiedGroupKFold (Unseen Local Authorities, Balanced Exclusion Split)
run_validated_cv(
    StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42),
    "Tuned Policy-Only: StratifiedGroupKFold (Unseen Counties)",
    use_groups=True
)


=== Tuned Policy-Only: Stratified 5-Fold CV ===
 Fold  Val_Exclusions  ROC_AUC  PR_AUC_Inclusion(1)  PR_AUC_Exclusion(0)
    1             108 0.976013             0.997583             0.841298
    2             108 0.978093             0.997855             0.785339
    3             107 0.972141             0.997166             0.813451
    4             107 0.955581             0.993891             0.797427
    5             107 0.976275             0.997718             0.794080
Mean Scores:
ROC_AUC                0.9716
PR_AUC_Inclusion(1)    0.9968
PR_AUC_Exclusion(0)    0.8063
Optimal Inclusion Cutoff: 0.2758 | Max Exclusion F1: 0.7779

=== Tuned Policy-Only: StratifiedGroupKFold (Unseen Counties) ===
 Fold  Val_Exclusions  ROC_AUC  PR_AUC_Inclusion(1)  PR_AUC_Exclusion(0)
    1              80 0.794313             0.979090             0.502961
    2              91 0.981415             0.998964             0.685216
    3              75 0.974911             0.997536             

np.float64(0.15935641527175903)

In [5]:
# ==============================================================================
# 4. FIT FINAL PRODUCTION MODEL ON 100% OF DATA & EXPORT BUNDLE
# ==============================================================================
final_ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_final = np.hstack([X_df[num_cols].values, final_ohe.fit_transform(X_df[cat_cols])])

final_model = XGBClassifier(
    **best_params,
    scale_pos_weight=global_spw,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1
)
final_model.fit(X_final, y)

os.makedirs("Models", exist_ok=True)
artifact_path = os.path.join("../Models", "RZLT_PolicyOnly_FinalModel.joblib")
joblib.dump({
    'model': final_model,
    'encoder': final_ohe,
    'num_cols': num_cols,
    'cat_cols': cat_cols,
    'feature_names': num_cols + list(final_ohe.get_feature_names_out(cat_cols)),
    'optimal_inclusion_threshold': float(opt_threshold),
    'best_params': best_params
}, artifact_path)

print(f"\nProduction model bundle saved -> {artifact_path}")


Production model bundle saved -> ../Models\RZLT_PolicyOnly_FinalModel.joblib


In [ ]:
# 1. Load saved production model bundle
bundle = joblib.load(os.path.join("../Models", "RZLT_PolicyOnly_FinalModel.joblib"))
model = bundle['model']
encoder = bundle['encoder']
num_cols = bundle['num_cols']
cat_cols = bundle['cat_cols']
threshold = bundle['optimal_inclusion_threshold']

# 2. Load dataset and score all parcels
gdf = gpd.read_file(os.path.join("../Datasets/RZLT_Samples", "RZLT_AllCodes_10to1_FullFeatures.geojson"))
gdf['ZONE_GZT'] = gdf['ZONE_GZT'].fillna("Unknown").astype(str).str.strip()

X_inf = np.hstack([gdf[num_cols].values, encoder.transform(gdf[cat_cols])])
gdf['PRED_INCLUSION_PROB'] = model.predict_proba(X_inf)[:, 1]
gdf['PRED_TARGET'] = (gdf['PRED_INCLUSION_PROB'] >= threshold).astype(int)

# 3. Inspect Highest-Confidence Disagreements (Policy Edge Cases)
audit_cols = [
    'PARCEL_ID', 'LOCAL_AUTHORITY', 'ZONE_GZT', 'SITE_AREA',
    'SHAPE_COMPACTNESS', 'TRUE_PARCELS_WITHIN_500M', 'DIST_TO_BUS_M',
    'DECISION_STATUS', 'PRED_INCLUSION_PROB'
]
audit_cols = [c for c in audit_cols if c in gdf.columns]

print("=== Top 5 Statutory Outliers: Predicted Included, Actually Excluded ===")
false_inclusions = gdf[(gdf['target'] == 0) & (gdf['PRED_TARGET'] == 1)].sort_values(
    by='PRED_INCLUSION_PROB', ascending=False
)
print(false_inclusions[audit_cols].head(5).to_string(index=False))

print("\n=== Top 5 Statutory Outliers: Predicted Excluded, Actually Included ===")
false_exclusions = gdf[(gdf['target'] == 1) & (gdf['PRED_TARGET'] == 0)].sort_values(
    by='PRED_INCLUSION_PROB', ascending=True
)
print(false_exclusions[audit_cols].head(5).to_string(index=False))